# 06B 关闭程序，再继续任务

先做一次干净退出，再人为制造“工具已执行但结果尚未保存”的中断。二者不同，恢复方式也不能相同。

本份实验的模型动作使用教学脚本或回放，但每次 MCP 文件工具都实际运行。真实 DeepSeek 的跨进程记录在 06C。

**本实践在总览中的位置**：把 06A 保存的状态放进“模型调用—工具执行—检查点提交”的生命周期，观察干净恢复、不确定窗口、预算和权限怎样共同决定下一步。

**贯穿案例**

模拟服务在工具已经返回、结果尚未写入检查点时中断：恢复程序无法凭空知道这次动作是否完成。两次独立进程验证正常续跑；故障注入则把任务标为待审阅，由 `resolve_uncertain` 明确决定是否重试，并记录预算中的尝试次数与真正保存的结果数。


## 学习路线与配套课件

本 Notebook 可以独立从头运行。先读定义和输入，预测结果，再执行并解释差异。小算例帮助理解机制；真实训练、工具执行与保存的 API 记录会分别标明来源。

本节重点：不确定动作重试后，为什么次数是 2、结果只有 1。

课件页：L06.04-S01、L06.04-S03。

沿原有课程单元顺序完成实验；新增对照放在相关代码后。练习先自行作答，展开参考分析后再比较。

In [1]:
from pathlib import Path
import sys, json, asyncio, tempfile, copy, inspect, subprocess, hashlib
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
CHAPTER = ROOT / "06 Harness 与持续任务运行"
sys.path.insert(0, str(CHAPTER / "代码"))
from harness_lab import Harness, SimulatedCrash, source_snapshot
from checkpoint_store import CheckpointStore, CheckpointConflict, encode
from runtime_events import EventLog
from agent_lab import ReplayModel, tool_response
from skill_lab import SkillLibrary
from IPython.display import display, Code
print("随课只读工具；SQLite 保存任务；JSONL 保存诊断事件。")


随课只读工具；SQLite 保存任务；JSONL 保存诊断事件。


## 1. 一步不是一次完整任务

运行时有四个主要阶段：

- model_ready：准备构建上下文并请求模型。
- model_inflight：已经预留请求次数，尚未保存响应。
- tools_ready：模型提出的动作已保存，接下来执行某个工具。
- tool_inflight：已经预留工具次数，尚未保存结果。

一轮可能有多个工具；next_tool 标记这组中下一项。只有结果完整后，才能构建下一次模型请求。下面阅读实际 step 实现，重点找两处“先保存 in-flight，再做外部动作”。


In [2]:
display(Code(inspect.getsource(Harness._step), language="python"))
# 完整实现位于 src/harness_lab.py；课堂分成模型分支与工具分支两段讲。


async def _step(self,model,crash_at=None):
        if self.state['status']!='active':return self.state
        if self.state['phase'] in {'model_inflight','tool_inflight'}:
            return self.stop('needs_review')
        if self.state['source_snapshot']!=self.runtime.snapshot():return self.stop('source_changed')
        library=self.runtime.library()
        for name in self.state['active_skills']:library.activate(name)
        async with self.runtime.connection() as client:
            tools=(await client.list_tools()).tools
            remote_names={t.name for t in tools}
            declarations=self.runtime.declarations(tools)
            if self.state['phase']=='model_ready':
                if self.state['turns']>=self.state['limits']['max_turns']:return self.stop('turn_budget')
                try:messages,context_report=self.build_context(library)
                except ValueError:return self.stop('context_budget')
                self.state['last_context_report']=context_report
                self.state['last_model_input']=self.store.put_artifact(self.run_id,{'messages':messages,'tools':declarations})
                self.state['turns']+=1;self.state['phase']='model_inflight';self.checkpoint()
                span=self.log.emit('course.model.started','started',metrics={'request_ordinal':self.state['turns']})
                start=time.perf_counter();usage_start=len(getattr(model,'usage',[]))
                try:response=await asyncio.to_thread(model,messages,declarations)
                except ModelError:
                    self.log.emit('course.model.failed','failed',span,error={'category':'model','type':'ModelError','retryable':False})
                    return self.stop('needs_review')
                if crash_at=='after_model':raise SimulatedCrash('模型已返回，尚未保存响应')
                self.state['usage']+=getattr(model,'usage',[])[usage_start:]
                self.log.emit('course.model.completed','succeeded',span,metrics={'duration_ms':round((time.perf_counter()-start)*1000,3)})
                calls=response.get('tool_calls') or []
                if not isinstance(calls,list) or any(not isinstance(c,dict) or not isinstance(c.get('function'),dict)
                    or not isinstance(c['function'].get('name'),str) or not isinstance(c['function'].get('arguments'),str) for c in calls):
                    self.state['phase']='finished';return self.stop('invalid_model_output')
                if not calls:
                    self.state['answer']=response.get('content') or '';self.state['phase']='finished'
                    return self.stop('final_answer' if self.state['answer'].strip() else 'empty_answer')
                ids=[c.get('id') for c in calls]
                if any(not isinstance(i,str) or not i for i in ids) or len(set(ids))!=len(ids) or set(ids).intersection(self.state['seen_ids']):
                    self.state['phase']='finished';return self.stop('invalid_call_ids')
                self.state['seen_ids']+=ids;self.state['messages'].append(response)
                self.state['pending']=calls;self.state['next_tool']=0;self.state['phase']='tools_ready';self.checkpoint()
                return self.state
            if self.state['phase']=='tools_ready':
                if self.state['tool_calls']>=self.state['limits']['max_tools']:return self.stop('tool_budget')
                call=self.state['pending'][self.state['next_tool']];fn=call.get('function',{})
                name=fn.get('name','');allowed=name in self.runtime.allowed
                self.log.emit('course.permission.completed','succeeded',attributes={'tool_name':name if allowed else '<unregistered>','decision':'allow' if allowed else 'deny'})
                self.state['tool_calls']+=1;self.state['phase']='tool_inflight';self.checkpoint()
                span=self.log.emit('course.tool.started','started',attributes={'tool_name':name if allowed else '<unregistered>'})
                start=time.perf_counter()
                try:
 

## 2. 两次独立进程，从同一任务继续

下面用 subprocess 启动随课 CLI。第一次推进三步：模型提出激活、执行激活、模型提出下一动作，然后退出。第二次使用同一数据库与 run_id 继续。

固定动作表会依次加载方法、参考、列文件、读入口和配置。这个表是教学脚本，不是模型推理。程序没有任意 shell 参数，子进程也不等于沙箱。


In [3]:
directory = ROOT / ".local/student-runs/l06/nb-process"
directory.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-X', 'utf8', str(CHAPTER / "代码/harness_cli.py"), "--db", str(directory / "runs.sqlite3")]
first = json.loads(subprocess.check_output(command + ["--goal", "观察恢复流程", "--steps", "3"], text=True, encoding='utf-8'))
print("第一个进程已退出:", first["state"]["phase"], "工具次数:", first["state"]["tool_calls"])
second = json.loads(subprocess.check_output(command + ["--run-id", first["run_id"]], text=True, encoding='utf-8'))
print("第二个进程:", second["state"]["status"], "工具次数:", second["state"]["tool_calls"])
assert first["run_id"] == second["run_id"]
assert second["state"]["tool_calls"] == 5
ids = [a["call_id"] for a in second["state"]["artifacts"]]
assert len(ids) == len(set(ids))
print("已经保存的激活动作未重复执行。")
process_evidence = {
    "run_id": first["run_id"],
    "first_process_phase": first["state"]["phase"],
    "final_status": second["state"]["status"],
    "final_tool_calls": second["state"]["tool_calls"],
    "duplicate_call_ids": len(ids) != len(set(ids)),
}


第一个进程已退出: tools_ready 工具次数: 1


第二个进程: final_answer 工具次数: 5
已经保存的激活动作未重复执行。


## 3. 主动暂停与恢复

暂停发生在一个已保存的安全边界。resume 只是让该任务再次进入 active 状态，不会凭空完成目标，也不会重置预算。

学习时可以用它检查下一步；生产应用是否需要审批，应由具体动作和既有授权决定，不应让所有只读调用都多一次无意义确认。


In [4]:
temporary = tempfile.TemporaryDirectory()
work = Path(temporary.name)
store = CheckpointStore(work / "runs.sqlite3")
h = Harness.create(store, "暂停示例", work / "events.jsonl")
h.pause()
reopened = Harness(store, h.run_id, work / "events.jsonl")
print("重新读取:", reopened.state["status"])
reopened.resume()
print("恢复:", reopened.state["status"], "已用请求:", reopened.state["turns"])


重新读取: paused
恢复: active 已用请求: 0


## 4. 最关键的反例：结果尚未落盘

我们让真实读文件工具返回后，主动抛出 SimulatedCrash，模拟尚未保存结果就被中断。它不是实际杀进程；刚才的 CLI 实验才是真正跨进程。

数据库只能看到 tool_inflight，无法确定外部动作是否发生。因此恢复先进入 needs_review，不自动再执行。只读动作可以由操作者显式决定重试；外部写入可能需要查询结果、幂等键或人工处理，不能照抄这一重试策略。


In [5]:
h = Harness.create(store, "读取配置并停顿", work / "events.jsonl")
model = ReplayModel([tool_response("read-1", "mcp_read_file", {"path":"config.py"}),
                     {"role":"assistant", "content":"回放完成"}])
await h.step(model)  # 先保存模型提出的动作
try:
    await h.step(model, crash_at="after_tool")
except SimulatedCrash as error:
    print("课堂注入中断:", error)
recovered = Harness(store, h.run_id, work / "events.jsonl")
await recovered.step(model)
print("恢复后:", recovered.state["status"], recovered.state["phase"])
assert recovered.state["status"] == "needs_review"


课堂注入中断: 工具已返回，尚未保存结果
恢复后: needs_review tool_inflight


In [6]:
# 这条显式操作表示课堂操作者选择重试已知只读动作；不是等待超时自动批准。
recovered.resolve_uncertain("retry")
state = await recovered.drive(model)
print(state["status"], "已预留工具次数:", state["tool_calls"], "已保存结果数:", len(state["artifacts"]))
assert state["tool_calls"] == 2 and len(state["artifacts"]) == 1
# 第一次可能已经消耗资源，所以计数不退回。
uncertain_evidence = {
    "final_status": state["status"],
    "reserved_tool_attempts": state["tool_calls"],
    "saved_results": len(state["artifacts"]),
    "review_required_before_retry": True,
}


final_answer 已预留工具次数: 2 已保存结果数: 1


### 不确定动作重试后，为什么次数是 2、结果只有 1

配套课件：L06.04-S01、L06.04-S03。

刚才在工具已执行但结果未保存处注入了中断。恢复者只能知道动作进入了执行阶段，不能断言它没有发生。课堂操作者已对只读动作显式选择 retry；下面从恢复后状态核对预算与产物。

In [7]:
case_attempts = state["tool_calls"]
case_saved = len(state["artifacts"])
print("计入预算的动作尝试:", case_attempts)
print("成功保存的结果:", case_saved)
print("差值:", case_attempts - case_saved)
assert case_attempts == 2 and case_saved == 1
print("有产物的调用 ID:", [a["call_id"] for a in state["artifacts"]])


计入预算的动作尝试: 2
成功保存的结果: 1
差值: 1
有产物的调用 ID: ['read-1']


**结果解读**

第一次可能已经执行并消耗资源，因此计数不退回；重试产生了保存下来的结果。2 次尝试与 1 份产物都是真实状态，不能根据产物数覆盖预算计数，也不能据此宣称通用 exactly-once。

**小练习**

若动作变为“发送邮件”，为什么不能照搬本例自动重试？需要什么额外证据或机制？

<details><summary>完成后展开参考分析</summary>

发送可能已发生。需要操作状态查询、幂等键或人工确认等任务相关机制，不能把只读演示的选择推广到不可重复副作用。

</details>

公式与实现来源见 [配套素材来源](../../../references/companion-upgrade-sources.md)。

## 5. 预算跨恢复保留

重启不能把已经用掉的请求次数清零。尤其当模型返回后还没保存响应时，可能已经计费；本地缺少结果不等于请求没有发生。

下面设一轮模型预算，模拟响应未保存。显式选择重试后，预算检查仍应阻止第二次请求。


In [8]:
h = Harness.create(store, "一轮预算", work / "events.jsonl", max_turns=1)
try:
    await h.step(ReplayModel([{"role":"assistant","content":"返回但未保存"}]), crash_at="after_model")
except SimulatedCrash:
    pass
h = Harness(store, h.run_id, work / "events.jsonl")
h.resolve_uncertain("retry")
result = await h.drive(ReplayModel([]))
print(result["status"], result["turns"])
assert result["status"] == "turn_budget" and result["turns"] == 1
budget_evidence = {
    "status_after_restart": result["status"],
    "reserved_model_attempts": result["turns"],
    "budget_reset_on_restart": False,
}


turn_budget 1


## 6. 权限在实际执行前判断

模型可以提出不存在的写工具，应用不应该据此执行它。本课只允许加载教学方法、读参考、MCP 文件读取以及读取当前任务产物。工具参数和文件范围还要由具体实现校验。

练习：给模型一个 write_file 动作，检查产物中记录的是什么，是否真的修改过源码。


In [9]:
h = Harness.create(store, "边界练习", work / "events.jsonl")
before = source_snapshot()
model = ReplayModel([tool_response("write-1", "write_file", {"path":"config.py"}),
                     {"role":"assistant","content":"停止"}])
state = await h.drive(model)
observed = store.read_artifact(h.run_id, state["artifacts"][0]["id"])
print(observed)
assert source_snapshot() == before


{'error_type': 'PermissionError', 'message': '工具调用未完成', 'ok': False}


In [10]:
assert observed["ok"] is False and observed["error_type"] == "PermissionError"
print("提出写入不等于执行写入；这次只保存了拒绝结果。")


提出写入不等于执行写入；这次只保存了拒绝结果。


## 可带走的结果：中断恢复决策单

把三种恢复情形放在同一份报告中：干净检查点直接继续；结果不确定时先审核再决定；预算和权限在重启后仍然有效。报告同时保留“动作尝试次数”和“已保存结果数”，避免把二者误认为相同。


In [11]:
report_dir = ROOT / ".local/student-runs/l06"
report_dir.mkdir(parents=True, exist_ok=True)
report = {
    "report_type": "interrupt-recovery-decision",
    "clean_restart": process_evidence,
    "uncertain_tool_window": uncertain_evidence,
    "budget_continuity": budget_evidence,
    "permission_boundary": {
        "requested_tool": "write_file",
        "executed": False,
        "recorded_error": observed["error_type"],
        "source_unchanged": source_snapshot() == before,
    },
    "decision_rules": [
        "continue only from a committed checkpoint",
        "review an uncertain external action before retrying",
        "never reset consumed budget or inherited permission on restart",
    ],
}
report_path = report_dir / "interrupt-recovery-decision.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("已保存恢复决策单:", report_path.relative_to(ROOT))
print(json.dumps(report, ensure_ascii=False, indent=2))
store.close(); temporary.cleanup()


已保存恢复决策单: .local\student-runs\l06\interrupt-recovery-decision.json
{
  "report_type": "interrupt-recovery-decision",
  "clean_restart": {
    "run_id": "283898681b4446e383dd58a15d267c23",
    "first_process_phase": "tools_ready",
    "final_status": "final_answer",
    "final_tool_calls": 5,
    "duplicate_call_ids": false
  },
  "uncertain_tool_window": {
    "final_status": "final_answer",
    "reserved_tool_attempts": 2,
    "saved_results": 1,
    "review_required_before_retry": true
  },
  "budget_continuity": {
    "status_after_restart": "turn_budget",
    "reserved_model_attempts": 1,
    "budget_reset_on_restart": false
  },
  "permission_boundary": {
    "requested_tool": "write_file",
    "executed": false,
    "recorded_error": "PermissionError",
    "source_unchanged": true
  },
  "decision_rules": [
    "continue only from a committed checkpoint",
    "review an uncertain external action before retrying",
    "never reset consumed budget or inherited permission on restart

## 回顾

干净检查点可以继续；不确定的外部动作必须先判断；重启不重置预算；已保存的结果和权限决策可以核查。一次局部实验通过不代表任意外部写入都能“恰好一次”执行。

来源：[暂停与恢复的语义](https://docs.langchain.com/oss/python/langgraph/interrupts)。
